# Assignment 2 – Tweet Sentiment Classification
**Name:** Radiyah Shafiq **Roll No:** 23P-3017

**Instructions:**
- Answer each question in the code cell(s) directly below it.
- Do not change the question cells.
- Run all cells from top to bottom before submitting, so your outputs are visible.

### Import Libraries
Import all the libraries you need for this assignment here.

In [13]:
import re
import string
import numpy as np
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report
)


nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Libraries imported successfully.")

Libraries imported successfully.


---
## Question 1: Data Preparation
**Load the CSV file, remove the `tweet_id` column, and create a sentiment mapping dictionary where `'sadness'` → 0 and `'happiness'` → 1. Apply the mapping to the sentiment column.**

In [15]:
#Loading the dataset
df = pd.read_csv("dataset - dataset (1).csv")

#Removing tweet_id because it is not needed for sentiment prediction
df = df.drop(columns=["tweet_id"])

#Filling missing tweet text with empty strings
df["content"] = df["content"].fillna("")

#Converting sentiment labels into numeric values
sentiment_mapping = {
    "sadness": 0,
    "happiness": 1
}
df["sentiment"] = df["sentiment"].map(sentiment_mapping)

#Removing rows having labels other than sadness and happiness
df = df.dropna(subset=["sentiment"])
df["sentiment"] = df["sentiment"].astype(int)

#Printing dataset shape and checking sentiment counts
print("Dataset shape:", df.shape)
print("\nSentiment counts:")
print(df["sentiment"].value_counts())

#Displaying first five rows of the dataset
df.head()

Dataset shape: (10374, 2)

Sentiment counts:
sentiment
1    5209
0    5165
Name: count, dtype: int64


,sentiment,content
0,0,Layin n bed with a headache ughhhh...waitin o...
1,0,Funeral ceremony...gloomy friday...
2,0,"I should be sleep, but im not! thinking about ..."
3,0,@charviray Charlene my love. I miss you
4,0,@kelcouch I'm sorry at least it's Friday?


---
## Question 2: Train Test Split
**Split the data into 80% training and 20% testing sets using `train_test_split()` with `random_state=42`.**

In [5]:
#Separating input text and target labels
X = df["content"]
y = df["sentiment"]

#Splitting data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

#Printing number of training and testing samples
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8299
Testing samples: 2075


---
## Question 3: Text Preprocessing
**Implement the following 6 functions:**
- `lower_case(text)`: Convert text to lowercase
- `removing_urls(text)`: Remove URLs using regex pattern matching
- `removing_punctuations(text)`: Remove special characters and punctuation
- `removing_numbers(text)`: Remove all numeric digits
- `remove_stop_words(text)`: Remove English stopwords using NLTK
- `lemmatization(text)`: Apply lemmatization using WordNetLemmatizer

In [16]:
#1.Converting text into lowercase
def lower_case(text):
    return str(text).lower()


#2.Removing URLs using regular expressions
def removing_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", "", str(text))


#3.Removing punctuation and special characters
def removing_punctuations(text):
    return re.sub(r"[^a-zA-Z\s]", " ", str(text))


#4.Removing numeric digits
def removing_numbers(text):
    return re.sub(r"\d+", "", str(text))


#Preparing English stopwords from NLTK
try:
    english_stopwords = set(stopwords.words("english"))
except LookupError:
    #Using sklearn stopwords if NLTK stopwords are not available
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    english_stopwords = set(ENGLISH_STOP_WORDS)


#5.Removing English stopwords
def remove_stop_words(text):
    words = str(text).split()
    filtered_words = [word for word in words if word not in english_stopwords]
    return " ".join(filtered_words)


#Creating the WordNet lemmatizer
lemmatizer = WordNetLemmatizer()


#6.Applying lemmatization using WordNetLemmatizer
def lemmatization(text):
    words = str(text).split()
    lemmatized_words = []

    for word in words:
        try:
            lemmatized_words.append(lemmatizer.lemmatize(word))
        except LookupError:
            #Keeping the word unchanged if WordNet is not available
            lemmatized_words.append(word)

    return " ".join(lemmatized_words)


#Testing the preprocessing functions with sample text
sample_text = "I am HAPPY! Visit https://example.com on 2025."
print("Original:", sample_text)
print("Lowercase:", lower_case(sample_text))
print("Without URLs:", removing_urls(lower_case(sample_text)))

Original: I am HAPPY! Visit https://example.com on 2025.
Lowercase: i am happy! visit https://example.com on 2025.
Without URLs: i am happy! visit  on 2025.


---
## Question 4: Normalize Text
**Create a `normalize_text(df)` function that applies all preprocessing functions in the correct order (lowercase → remove URLs → remove punctuation → remove numbers → remove stopwords → lemmatization). Apply this function to both training and testing datasets. After this, convert training and testing data into `X_train`, `y_train`, `X_test`, `y_test` as numpy arrays.**

In [17]:
def normalize_text(df):
    #Working on a copy to keep the original dataframe unchanged
    normalized_df = df.copy()

    #Applying all preprocessing functions in the required order
    normalized_df["content"] = normalized_df["content"].apply(lower_case)
    normalized_df["content"] = normalized_df["content"].apply(removing_urls)
    normalized_df["content"] = normalized_df["content"].apply(removing_punctuations)
    normalized_df["content"] = normalized_df["content"].apply(removing_numbers)
    normalized_df["content"] = normalized_df["content"].apply(remove_stop_words)
    normalized_df["content"] = normalized_df["content"].apply(lemmatization)

    return normalized_df


#Applying normalization on training and testing data
train_df = pd.DataFrame({
    "content": X_train,
    "sentiment": y_train
})
test_df = pd.DataFrame({
    "content": X_test,
    "sentiment": y_test
})

train_df = normalize_text(train_df)
test_df = normalize_text(test_df)

#Converting data into NumPy arrays
X_train = train_df["content"].to_numpy()
y_train = train_df["sentiment"].to_numpy()
X_test = test_df["content"].to_numpy()
y_test = test_df["sentiment"].to_numpy()

#Printing shapes to check the data
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nSample normalized tweet:", X_train[0])

X_train shape: (8299,)
y_train shape: (8299,)
X_test shape: (2075,)
y_test shape: (2075,)

Sample normalized tweet: quot problem miss cause quot


In [8]:
# Apply normalize_text to training and testing data and convert to numpy arrays

---
## Question 5: Feature Engineering
**Use `CountVectorizer` from scikit-learn to convert text to Bag of Words features.**

In [22]:
#Creating a new Bag of Words vectorizer
vectorizer = CountVectorizer()

#Learning vocabulary from training data
X_train_bow = vectorizer.fit_transform(X_train)

#Using the same vocabulary for testing data
X_test_bow = vectorizer.transform(X_test)

#Checking feature dimensions
print("Training shape:", X_train_bow.shape)
print("Testing shape:", X_test_bow.shape)

Training shape: (8299, 13741)
Testing shape: (2075, 13741)


---
## Question 6: Model Building
**Use `RandomForestClassifier` from scikit-learn to train the model.**

In [23]:

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train_bow, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

---
## Question 7: Model Evaluation
**Make predictions on test data and calculate the following metrics: Accuracy, Precision, Recall, AUC-ROC score. Also print the Classification Report.**

In [24]:
#Predicting sentiment labels for test data
y_pred = random_forest_model.predict(X_test_bow)

#Calculating evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

#Calculating AUC-ROC using happiness class probabilities
y_probability = random_forest_model.predict_proba(X_test_bow)[:, 1]
auc_roc = roc_auc_score(y_test, y_probability)

#Printing evaluation results
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("AUC-ROC:", round(auc_roc, 4))

#Displaying classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["sadness", "happiness"],
    zero_division=0
))

Accuracy: 0.7619
Precision: 0.7757
Recall: 0.7222
AUC-ROC: 0.8367

Classification Report:
              precision    recall  f1-score   support

     sadness       0.75      0.80      0.77      1060
   happiness       0.78      0.72      0.75      1015

    accuracy                           0.76      2075
   macro avg       0.76      0.76      0.76      2075
weighted avg       0.76      0.76      0.76      2075

